# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import os
from typing import cast
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
print(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
print(f"✅ Torch CUDA available: {cuda_test}")
device_name = torch.cuda.get_device_name(0)
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device Name: {device_name} | Device reference: {torch_device.type}")

### machine learning (scikit-learn)
import numpy as np
import pandas as pd
import torch
from typing import cast
from sklearn.pipeline import Pipeline
from tsfm_public import (
    TimeSeriesForecastingPipeline,
    TinyTimeMixerForPrediction,
)
from tsfm_public.toolkit.visualization import plot_predictions

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.preprocessing_project_specific as pps
import smartcheck.deep_learning_project_specific as dlps
import smartcheck.modeling_project_specific as mps

# 2. Loading and Preprocessing

## 2.1 Loading data

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Data Refactoring pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    # "date_et_heure_de_comptage",
    "date_et_heure_de_comptage_local",
    # "date_et_heure_de_comptage_utc",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("convert_datetime", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage",
                                                                   for_sarimax=True)),
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

# 3. Regression modeling

## 3.1 Contextual variables

#### For the model

In [ ]:
timestamp_column = "date_et_heure_de_comptage_local"
target_columns = ["comptage_horaire"]
context_length = 512
prediction_length = 96
batch_size: int = 96
# Output directory for writing evaluation results.
OUT_DIR = "ttm_results.model"

#### For the experiments

In [ ]:
dict_compteurs = {
    "experiment_1": {
        "key": ('135 avenue Daumesnil','SE-NO'),
        "name": "Daumesnil_S-N",
        "sub_range": (0,),
        "best_checkpoint": "checkpoint-410",
    },
    # "experiment_2": {
    #     "key": ('102 boulevard de Magenta', 'SE-NO'),
    #     "name": "Magenta-O-E",
    #     "sub_range": (0,),
    #     "best_checkpoint": "checkpoint-3",
    # },
    # "experiment_3": {
    #     "key": ('Totem 73 boulevard de Sébastopol', 'S-N'),
    #     "name": "Sébastopol_S-N",
    #     "sub_range": (0,),
    #     # "best_checkpoint": "checkpoint-248",
    # },
}

## 3.2 Data Viz of Time Series per counter

In [ ]:
grouped_df = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])

for experiment, exp_params in dict_compteurs.items():
    key = exp_params["key"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {exp_params} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue

    df_compteur = df_compteur.sort_values(by=timestamp_column)
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_compteur_sub = df_compteur[range_start:range_end]

    fig, axs = plt.subplots(len(target_columns), 1, figsize=(10, 2 * len(target_columns)), squeeze=False)
    for ax, target_column in zip(axs, target_columns):
        ax[0].plot(df_compteur_sub[timestamp_column], df_compteur_sub[target_column])
    plt.show()

## 3.3 Predictions

#### Calculation from Experiment dictionnary and Disk Context only

In [ ]:
for experiment, exp_params in dict_compteurs.items():
    # Extract the artifacts from the saved results
    key = exp_params["key"]
    name = exp_params["name"]
    best_checkpoint = exp_params["best_checkpoint"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {key} {name} {best_checkpoint} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue
    checkpoint_dir = os.path.join(OUT_DIR, f"output_{name}", best_checkpoint)
    preproc_dir = os.path.join(OUT_DIR, f"preproc_{name}")
    
    # Reload from disk for safe check
    reloaded_tsp = dlps.load_preprocessor_state(preproc_dir)
    reloaded_model = dlps.load_model_from_checkpoint(TinyTimeMixerForPrediction, checkpoint_dir, torch_device.type)
    logging.info(f"Bascule du modèle sur {torch_device}")
    reloaded_model.to(torch_device)  # type: ignore

    # Create the evaluation pipeline
    pipeline = TimeSeriesForecastingPipeline(
        model=reloaded_model,
        device=torch_device,
        feature_extractor=reloaded_tsp,
        batch_size=batch_size,
    )

    df_train, df_test = dlps.df_train_test_split_time_aware(
        df=df_compteur,
        timestamp_column=timestamp_column,
        test_size=0.25
    )
    # Collect the test df and run the predictions for train and test
    predictions_df_train = cast(pd.Dataframe, pipeline(df_train))  # type: ignore
    predictions_df_test = cast(pd.Dataframe, pipeline(df_test))  # type: ignore

    # Print/Plot the predictions
    plot_predictions(
        input_df=df_test,
        predictions_df=predictions_df_test,  # type: ignore
        freq="h",
        timestamp_column=timestamp_column,
        channel=target_column,
        # we check the prediction on each of the 3 previous week and in the future
        indices=[ -24*7*3, -24*7*2, -24*7*1, -1],
        num_plots=4,
    )
    plt.show()

    ###############################################
    ### Stockage des données pour les métriques ###
    ###############################################
    dict_compteurs[experiment]["y_train"] = predictions_df_train.comptage_horaire.apply(
        lambda x: x[0]
    ).iloc[:-1]
    dict_compteurs[experiment]["y_train_pred"] = predictions_df_train.comptage_horaire_prediction.apply(
        lambda x: x[0]
    ).iloc[:-1]
    dict_compteurs[experiment]["y_test"] = predictions_df_test.comptage_horaire.apply(
        lambda x: x[0]
    ).iloc[:-1]
    dict_compteurs[experiment]["y_test_pred"] = predictions_df_test.comptage_horaire_prediction.apply(
        lambda x: x[0]
    ).iloc[:-1]
    dict_compteurs[experiment]["dates_test"] = predictions_df_test[["date_et_heure_de_comptage_local"]].iloc[:-1]

In [ ]:
for experiment, exp_params in dict_compteurs.items():
    # Extract the artifacts from the saved results
    key = exp_params["key"]
    name = exp_params["name"]
    best_checkpoint = exp_params["best_checkpoint"]
    if key in grouped_df.groups:
        df_compteur = grouped_df.get_group(key)
        logging.info(f"\n--- {experiment} : {key} {name} {best_checkpoint} ---")
    else:
        logging.info(f"⚠️ Clé {key} non trouvée dans les groupes de df.")
        continue
    checkpoint_dir = os.path.join(OUT_DIR, f"output_{name}", best_checkpoint)
    preproc_dir = os.path.join(OUT_DIR, f"preproc_{name}")

    y_train = exp_params["y_train"]
    y_train_pred = exp_params["y_train_pred"]
    y_test = exp_params["y_test"]
    y_test_pred = exp_params["y_test_pred"]
    dates_test = exp_params["dates_test"]
    periode_limite = (
        dates_test.date_et_heure_de_comptage_local[len(dates_test)-24*7*4],
        dates_test.date_et_heure_de_comptage_local[len(dates_test)-1], # 5 dernière semaines
    )

    # Affichage des metrique train et test
    model_train_metrics = mps.compute_metrics(
        y_train,
        y_train_pred,
    )
    model_test_metrics = mps.compute_metrics(
        y_test,
        y_test_pred
    )
    logging.info(f"Metriques du modèle (Train): {model_train_metrics}")
    logging.info(f"Metriques du modèle (Test): {model_test_metrics}")

    # projection des predictions de test dans le temps
    fig_pred = mps.plot_predictions(
        str(key),
        dates_test, 
        y_test, 
        y_test_pred, 
        periode_limite=periode_limite,
    )
    plt.show()

    # projection des résidus et calcul du coefficient de dérive dans le temps
    fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
        str(key),
        dates_test, 
        y_test, 
        y_test_pred.values, 
        periode_limite=periode_limite
    )
    logging.info(f"Pente de la droite de régression des résidus dans le temps (dérive) : {model_res_coeff}")
    plt.show()